# <font color="#418FDE" size="6.5" uppercase>**Klassifikation mit NumPy**</font>

>Last update: 20260723.
    
By the end of this Lecture, you will be able to:
- Berechnen Klassifikationsscores, Wahrscheinlichkeiten und binäre Verluste. 
- Trainieren eine kleine logistische Regression mit NumPy. 
- Vergleichen logistische Regression, k-NN, Naive Bayes und Baselines. 


## **1. Lineare Klassifikation**

### **1.1. Perzeptron Schwelle**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_08/Lecture_A/image_01_01.jpg?v=1784792707" width="250">



>* Merkmale werden gewichtet und zu Scores kombiniert
>* Der Score wird mit einer Schwelle verglichen

>* Schwelle entscheidet zwischen positiver und negativer Klasse
>* Schwellenhöhe steuert übersehene Fälle und Fehlalarme

>* Perzeptron trennt Daten durch lineare Grenzen
>* Einfach, aber bei komplexen Mustern begrenzt



In [ ]:
#@title Python-Code - Perzeptron Schwelle

# Dieses Beispiel zeigt eine einfache Perzeptron-Schwelle.
# Ein linearer Score wird mit Schwellen verglichen.
# Die Ausgabe zeigt veränderte binäre Entscheidungen.

import numpy as np
import matplotlib.pyplot as plt

# Kleine Beispieldaten beschreiben E-Mails mit zwei Merkmalen.
features = np.array([[0, 1], [1, 1], [2, 1], [1, 3], [3, 2], [4, 3]])
labels = np.array([0, 0, 0, 1, 1, 1])

# Gewichte bestimmen, wie stark jedes Merkmal den Score beeinflusst.
weights = np.array([0.9, 1.2])
scores = features @ weights

# Zwei Schwellen zeigen, wie dieselben Scores anders entschieden werden.
low_threshold = 3.0
high_threshold = 5.0
low_predictions = (scores >= low_threshold).astype(int)
high_predictions = (scores >= high_threshold).astype(int)

# Eine kurze Prüfung schützt vor unpassenden Formen.
if features.shape[0] != labels.size:
    raise ValueError("Jeder Datenpunkt braucht genau ein Label.")

print("Perzeptron-Score = 0.9 * Links + 1.2 * Signalwörter")
print("Scores:", np.round(scores, 2).tolist())
print("Wahre Klassen:", labels.tolist())
print("Vorhersage bei Schwelle 3.0:", low_predictions.tolist())
print("Vorhersage bei Schwelle 5.0:", high_predictions.tolist())

# Die Grafik zeigt Scores und zwei mögliche Entscheidungsgrenzen.
fig, ax = plt.subplots(figsize=(7, 4))
point_numbers = np.arange(1, scores.size + 1)
ax.scatter(point_numbers, scores, c=labels, cmap="coolwarm", s=90, label="E-Mail")
ax.axhline(low_threshold, color="green", linestyle="--", label="Schwelle 3.0")
ax.axhline(high_threshold, color="black", linestyle=":", label="Schwelle 5.0")

ax.set_title("Perzeptron-Scores und Schwellenentscheidung")
ax.set_xlabel("Datenpunkt")
ax.set_ylabel("Linearer Score")
ax.set_xticks(point_numbers)
ax.legend()
plt.show()



### **1.2. Sigmoid und Logits**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_08/Lecture_A/image_01_02.jpg?v=1784792711" width="250">



>* Logit: roher Score aus Merkmalen und Gewichten
>* Vorzeichen und Größe zeigen Modell-Evidenz

>* Sigmoid macht Logits zu Wahrscheinlichkeiten
>* Wahrscheinlichkeiten zeigen die Sicherheit der Vorhersage

>* Logits rechnen, Wahrscheinlichkeiten erklären Entscheidungen
>* Schwellenwerte hängen vom Anwendungsrisiko ab



In [ ]:
#@title Python-Code - Sigmoid und Logits

# Dieses Beispiel verbindet Logits mit Wahrscheinlichkeiten.
# Die Sigmoid-Funktion macht Scores interpretierbar.
# Die Ausgabe zeigt Schwellen und Verluste.

import numpy as np
import matplotlib.pyplot as plt

# Kleine Logits zeigen typische Modell-Scores.
logits = np.array([-6.0, -2.0, 0.0, 2.0, 6.0])

# Die Sigmoid-Funktion wandelt jeden Logit um.
probabilities = 1.0 / (1.0 + np.exp(-logits))

# Ein Schwellenwert erzeugt binäre Klassenentscheidungen.
threshold = 0.5
predicted_classes = (probabilities >= threshold).astype(int)

# Beispiel-Labels erlauben einen einfachen binären Verlust.
true_labels = np.array([0, 0, 1, 1, 1])

# Kleine Begrenzung verhindert Logarithmus von null.
epsilon = 1e-12
safe_probabilities = np.clip(probabilities, epsilon, 1.0 - epsilon)

# Binary Cross-Entropy bestraft falsche sichere Vorhersagen stark.
losses = -(
    true_labels * np.log(safe_probabilities)
    + (1 - true_labels) * np.log(1 - safe_probabilities)
)

# Eine kurze Tabelle fasst die wichtigsten Werte zusammen.
print("Logit | Sigmoid | Klasse | Label | Verlust")
for logit, probability, predicted, label, loss in zip(
    logits, probabilities, predicted_classes, true_labels, losses
):
    print(
        f"{logit:5.1f} | {probability:7.3f} | {predicted:6d} | "
        f"{label:5d} | {loss:7.3f}"
    )

# Eine glatte Kurve zeigt die gesamte Transformation.
curve_logits = np.linspace(-8.0, 8.0, 200)
curve_probabilities = 1.0 / (1.0 + np.exp(-curve_logits))

# Die Grafik macht den Übergang bei Logit null sichtbar.
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(curve_logits, curve_probabilities, label="Sigmoid")
ax.scatter(logits, probabilities, color="red", label="Beispiele")
ax.axhline(threshold, color="gray", linestyle="--", label="Schwelle 0,5")
ax.set_title("Von Logits zu Wahrscheinlichkeiten")
ax.set_xlabel("Logit")
ax.set_ylabel("Wahrscheinlichkeit der positiven Klasse")
ax.legend()
plt.show()



### **1.3. Kreuzentropie Verlust**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_08/Lecture_A/image_01_03.jpg?v=1784792709" width="250">



>* Misst, wie gut Wahrscheinlichkeiten zu Klassen passen
>* Berücksichtigt auch die Sicherheit der Vorhersage

>* Bestraft sichere Fehlvorhersagen besonders stark
>* Fördert besser kalibrierte Wahrscheinlichkeiten

>* Kreuzentropie passt zu Sigmoid-Wahrscheinlichkeiten
>* Feines Lernsignal für bessere Gewichtsanpassung



In [ ]:
#@title Python-Code - Kreuzentropie Verlust

# Dieses Beispiel berechnet Kreuzentropie für binäre Klassifikation.
# Wahrscheinlichkeiten werden mit wahren Klassen verglichen.
# Selbstsichere Fehler erhalten besonders große Verluste.

import numpy as np

# Kleine Beispieldaten zeigen verschiedene Vorhersagequalitäten.
true_labels = np.array([1, 1, 0, 0, 1, 0])
predicted_probabilities = np.array([0.95, 0.55, 0.10, 0.80, 0.20, 0.45])

# Die Formprüfung verhindert unpassende Vergleiche.
if true_labels.shape != predicted_probabilities.shape:
    raise ValueError("Klassen und Wahrscheinlichkeiten müssen gleich lang sein.")

# Clipping schützt den Logarithmus vor null und eins.
epsilon = 1e-15
safe_probabilities = np.clip(predicted_probabilities, epsilon, 1 - epsilon)

# Kreuzentropie wird für jedes Beispiel einzeln berechnet.
losses = -(
    true_labels * np.log(safe_probabilities)
    + (1 - true_labels) * np.log(1 - safe_probabilities)
)

# Der Durchschnitt ist der Trainingsverlust für diese Mini-Daten.
mean_loss = np.mean(losses)
predicted_classes = (predicted_probabilities >= 0.5).astype(int)

# Die Ausgabe verbindet Wahrscheinlichkeit, Entscheidung und Verlust.
print("Beispiel | wahr | p(Klasse 1) | Vorhersage | Verlust")
for index in range(len(true_labels)):
    print(f"{index + 1:8d} | {true_labels[index]:4d} | {predicted_probabilities[index]:11.2f} | {predicted_classes[index]:10d} | {losses[index]:7.3f}")

# Der Mittelwert fasst alle Einzelverluste zusammen.
print(f"Durchschnittlicher Kreuzentropie Verlust: {mean_loss:.3f}")



## **2. Logistische Regression trainieren**

### **2.1. Gradienten berechnen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_08/Lecture_A/image_02_01.jpg?v=1784792703" width="250">



>* Gradienten zeigen die Richtung der Parameterkorrektur
>* Fehlerhafte Vorhersagen verändern passende Gewichte

>* Gradient misst Fehlerstärke der Vorhersage
>* Merkmale steuern Gewichtsänderungen vektorisiert in NumPy

>* Gradienten formen Gewichte und Bias schrittweise
>* NumPy macht die Lernbewegung nachvollziehbar



In [ ]:
#@title Python-Code - Gradienten berechnen

# Wir berechnen Gradienten für logistische Regression.
# Der Fehlervektor steuert Gewichte und Bias.
# Die Ausgabe zeigt eine kleine Parameterkorrektur.

import numpy as np

# Diese kleinen Daten haben zwei Merkmale und binäre Klassen.
X = np.array([[0.2, 1.0], [1.0, 0.3], [1.2, 1.1], [2.0, 1.5]])
y = np.array([0, 0, 1, 1])

# Startwerte sind absichtlich einfach und noch nicht gut angepasst.
weights = np.array([0.1, -0.2])
bias = 0.0
learning_rate = 0.5

# Die Sigmoidfunktion wandelt Scores in Wahrscheinlichkeiten um.
scores = X @ weights + bias
probabilities = 1 / (1 + np.exp(-scores))

# Der Fehlervektor ist Vorhersage minus tatsächliche Klasse.
errors = probabilities - y
n_samples = X.shape[0]

# Matrixmultiplikation sammelt die Korrekturen für alle Gewichte.
grad_weights = X.T @ errors / n_samples
grad_bias = np.mean(errors)

# Ein Gradientenschritt bewegt sich entgegen der Verluststeigung.
new_weights = weights - learning_rate * grad_weights
new_bias = bias - learning_rate * grad_bias

# Diese Prüfung macht die erwarteten Formen ausdrücklich sichtbar.
if grad_weights.shape != weights.shape:
    raise ValueError("Die Gradientenform passt nicht zu den Gewichten.")

print("Wahrscheinlichkeiten:", np.round(probabilities, 3))
print("Fehlervektor:", np.round(errors, 3))
print("Gradient Gewichte:", np.round(grad_weights, 3))
print("Gradient Bias:", round(float(grad_bias), 3))
print("Neue Gewichte:", np.round(new_weights, 3))
print("Neuer Bias:", round(float(new_bias), 3))



### **2.2. Training mit NumPy**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_08/Lecture_A/image_02_02.jpg?v=1784792701" width="250">



>* Parameter lernen aus Vorhersagefehlern
>* NumPy verarbeitet viele Beispiele effizient

>* Scores, Verlust und Gradienten wiederholt berechnen
>* Lernrate wählen und sinkenden Verlust prüfen

>* NumPy macht Lernschritte und Parameter sichtbar
>* Logistische Regression ist interpretierbar, aber begrenzt



In [ ]:
#@title Python-Code - Training mit NumPy

# Wir trainieren logistische Regression nur mit NumPy.
# Gradientenabstieg verbessert Gewichte und Bias schrittweise.
# Der Verlust sinkt und die Genauigkeit steigt.

import numpy as np
import matplotlib.pyplot as plt

# Diese kleinen Daten sind ungefähr linear trennbar.
X = np.array([[0.2, 1.1], [0.8, 1.4], [1.2, 0.7], [1.8, 1.0],
              [2.2, 2.0], [2.8, 2.4], [3.2, 2.1], [3.8, 2.8]])
y = np.array([0, 0, 0, 0, 1, 1, 1, 1])

# Die Formprüfung verhindert stille Rechenfehler.
if X.shape[0] != y.shape[0]:
    raise ValueError("Jede Zeile braucht genau ein Klassenlabel.")

# Standardisierung macht die Lernschritte stabiler.
feature_mean = X.mean(axis=0)
feature_std = X.std(axis=0)
X_scaled = (X - feature_mean) / feature_std

# Gewichte und Bias starten bewusst bei null.
weights = np.zeros(X_scaled.shape[1])
bias = 0.0
learning_rate = 0.4

# Diese Listen speichern den Lernverlauf.
loss_history = []
accuracy_history = []

# Die Sigmoidfunktion wandelt Scores in Wahrscheinlichkeiten um.
for step in range(80):
    scores = X_scaled @ weights + bias
    probabilities = 1.0 / (1.0 + np.exp(-scores))

    # Der binäre Log-Loss bestraft sichere falsche Vorhersagen stark.
    clipped = np.clip(probabilities, 1e-9, 1.0 - 1e-9)
    loss = -np.mean(y * np.log(clipped) + (1 - y) * np.log(1 - clipped))
    loss_history.append(loss)

    # Gradienten zeigen die Richtung der stärksten Verlustzunahme.
    errors = probabilities - y
    weight_gradient = X_scaled.T @ errors / len(y)
    bias_gradient = errors.mean()

    # Wir gehen entgegengesetzt zum Gradienten.
    weights = weights - learning_rate * weight_gradient
    bias = bias - learning_rate * bias_gradient

    # Die Genauigkeit zeigt die aktuelle Klassifikationsleistung.
    predictions = (probabilities >= 0.5).astype(int)
    accuracy = np.mean(predictions == y)
    accuracy_history.append(accuracy)

# Nach dem Training berechnen wir finale Vorhersagen.
final_scores = X_scaled @ weights + bias
final_probabilities = 1.0 / (1.0 + np.exp(-final_scores))
final_predictions = (final_probabilities >= 0.5).astype(int)

# Kurze Ausgaben verbinden Parameter, Verlust und Ergebnis.
print(f"Startverlust: {loss_history[0]:.3f}")
print(f"Endverlust: {loss_history[-1]:.3f}")
print(f"Endgenauigkeit: {np.mean(final_predictions == y):.2f}")
print(f"Gelernte Gewichte: {np.round(weights, 2)}")
print(f"Gelernter Bias: {bias:.2f}")

# Die Kurve macht den Trainingseffekt sichtbar.
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(loss_history, label="Log-Loss")
ax.set_title("Training einer logistischen Regression mit NumPy")
ax.set_xlabel("Trainingsschritt")
ax.set_ylabel("Verlust")
ax.legend()
plt.show()



### **2.3. Entscheidungsgrenze verstehen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_08/Lecture_A/image_02_03.jpg?v=1784792704" width="250">



>* Grenze zeigt den Klassenwechsel im Merkmalsraum
>* Scores werden zu Wahrscheinlichkeiten und Klassen

>* Lineare Grenze trennt Klassen im Merkmalsraum
>* Gewichte und Bias bestimmen ihre Lage

>* Lineare Grenzen funktionieren bei einfacher Trennung
>* Komplexe Muster erfordern oft andere Modelle



In [ ]:
#@title Python-Code - Entscheidungsgrenze verstehen

# Wir visualisieren eine gelernte logistische Entscheidungsgrenze.
# Die Grenze entsteht bei Wahrscheinlichkeit genau 0,5.
# Punkte auf beiden Seiten erhalten verschiedene Klassen.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import sklearn

# Wir erzeugen kleine zweidimensionale Klassifikationsdaten.
features, labels = make_classification(
    n_samples=120,
    n_features=2,
    n_redundant=0,
    n_informative=2,
    n_clusters_per_class=1,
    class_sep=1.4,
    random_state=42,
)

# Diese Prüfung macht die erwartete Datenform sichtbar.
if features.shape != (120, 2):
    raise ValueError("Die Beispieldaten haben nicht die erwartete Form.")

# Das Modell lernt Gewichte und Bias aus den Daten.
model = LogisticRegression(random_state=42, max_iter=200)
model.fit(features, labels)

# Die Genauigkeit zeigt kurz, ob die Grenze sinnvoll trennt.
predicted_labels = model.predict(features)
accuracy = accuracy_score(labels, predicted_labels)
print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Trainingsgenauigkeit: {accuracy:.2f}")
print("Entscheidungsgrenze: Score = 0, also Wahrscheinlichkeit = 0,5")

# Für die Gerade lösen wir w0*x + w1*y + b = 0.
weight_x = model.coef_[0, 0]
weight_y = model.coef_[0, 1]
bias = model.intercept_[0]

# Eine fast senkrechte Grenze würde diese Formel instabil machen.
if abs(weight_y) < 1e-8:
    raise ValueError("Die Grenze ist für diese einfache Zeichnung zu steil.")

# Wir berechnen passende x-Werte und die Grenzlinie.
x_values = np.linspace(features[:, 0].min() - 0.5, features[:, 0].max() + 0.5, 100)
y_values = -(weight_x * x_values + bias) / weight_y

# Die Grafik zeigt Datenpunkte und gelernte Entscheidungsgrenze.
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(
    features[:, 0],
    features[:, 1],
    c=labels,
    cmap="coolwarm",
    edgecolor="black",
)

# Die schwarze Linie markiert die Modellunsicherheit.
ax.plot(x_values, y_values, color="black", linewidth=2, label="Grenze bei 0,5")
ax.set_title("Logistische Regression: Entscheidungsgrenze")
ax.set_xlabel("Merkmal 1")
ax.set_ylabel("Merkmal 2")
ax.legend()
plt.show()



## **3. Einfache Modellvergleiche**

### **3.1. k nächste Nachbarn**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_08/Lecture_A/image_03_01.jpg?v=1784792713" width="250">



>* Neue Punkte übernehmen häufigste Nachbarklasse
>* Trainingsdaten dienen als gespeicherte Vergleichsbasis

>* k-NN erkennt flexible, nichtlineare Klassengrenzen
>* Skalierung, Distanzmaß und k sorgfältig wählen

>* k-NN ist verständlich, aber rechenintensiv
>* Vergleiche Genauigkeit, Stabilität und echten Mehrwert



In [ ]:
#@title Python-Code - k nächste Nachbarn

# Dieses Beispiel vergleicht k-NN mit einer einfachen Basislinie.
# k-NN nutzt Nachbarschaften statt gelernter Modellgewichte.
# Die Ausgabe zeigt Genauigkeit und eine einzelne Vorhersage.

import numpy as np
import sklearn
from sklearn.datasets import load_iris
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

# Wir verwenden einen kleinen, eingebauten Klassifikationsdatensatz.
iris = load_iris()
X = iris.data[:, :2]
y = iris.target

# Diese Prüfung macht die erwartete Tabellenform sichtbar.
if X.shape[0] != y.shape[0]:
    raise ValueError("Merkmale und Zielwerte passen nicht zusammen.")

# Die Aufteilung bleibt durch random_state reproduzierbar.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

# Skalierung wird nur auf den Trainingsdaten gelernt.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# k-NN entscheidet über die Mehrheit der nächsten Nachbarn.
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)

# Die Basislinie rät immer die häufigste Trainingsklasse.
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train_scaled, y_train)

# Beide Modelle werden auf denselben Testdaten bewertet.
knn_accuracy = accuracy_score(y_test, knn.predict(X_test_scaled))
baseline_accuracy = accuracy_score(y_test, baseline.predict(X_test_scaled))

# Ein einzelner Testpunkt zeigt die Nachbarschaftsidee konkret.
sample = X_test_scaled[:1]
distances, neighbor_indices = knn.kneighbors(sample)
neighbor_labels = y_train[neighbor_indices[0]]
predicted_label = knn.predict(sample)[0]

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"k-NN-Genauigkeit: {knn_accuracy:.2f}")
print(f"Basislinien-Genauigkeit: {baseline_accuracy:.2f}")
print(f"Nachbar-Klassen für einen Testpunkt: {neighbor_labels.tolist()}")
print(f"Vorhergesagte Klasse: {iris.target_names[predicted_label]}")



### **3.2. Naive Bayes**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_08/Lecture_A/image_03_02.jpg?v=1784792715" width="250">



>* Kombiniert Klassenhäufigkeit mit Merkmalswahrscheinlichkeiten
>* Einfache Annahmen, oft stark bei Spam

>* Schnell bei vielen Merkmalen, besonders Textdaten
>* Redundante Merkmale machen Wahrscheinlichkeiten oft übertrieben

>* Naive Bayes als starke einfache Referenz
>* Fair vergleichen mit passenden Bewertungsmaßen



In [ ]:
#@title Python-Code - Naive Bayes

# Dieses Beispiel vergleicht Naive Bayes mit Baselines.
# Wir nutzen kleine Iris-Daten für Klassifikation.
# Die Ausgabe zeigt Genauigkeit und einfache Einordnung.

import numpy as np
import sklearn
from sklearn.datasets import load_iris
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB

# Die Iris-Daten sind klein und bereits lokal verfügbar.
iris = load_iris()
X = iris.data
y = iris.target

# Eine kurze Prüfung verhindert verwirrende Formfehler.
if X.shape[0] != y.shape[0]:
    raise ValueError("Merkmale und Zielwerte passen nicht zusammen.")

# Die Aufteilung bleibt durch random_state reproduzierbar.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

# Die Baseline rät immer die häufigste Klasse im Training.
baseline_model = DummyClassifier(strategy="most_frequent")
baseline_model.fit(X_train, y_train)
baseline_predictions = baseline_model.predict(X_test)

# GaussianNB modelliert Merkmalsverteilungen pro Klasse.
nb_model = GaussianNB()
nb_model.fit(X_train, y_train)
nb_predictions = nb_model.predict(X_test)

# Wahrscheinlichkeiten zeigen, wie sicher das Modell wirkt.
first_probabilities = nb_model.predict_proba(X_test[:1])[0]
rounded_probabilities = np.round(first_probabilities, 3)

# Genauigkeit macht den einfachen Modellvergleich sichtbar.
baseline_accuracy = accuracy_score(y_test, baseline_predictions)
nb_accuracy = accuracy_score(y_test, nb_predictions)

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Baseline-Genauigkeit: {baseline_accuracy:.3f}")
print(f"Naive-Bayes-Genauigkeit: {nb_accuracy:.3f}")
print(f"Wahrscheinlichkeiten für ein Testbeispiel: {rounded_probabilities}")
print("Naive Bayes nutzt einfache Verteilungsannahmen statt nur Klassenhäufigkeit.")



### **3.3. Modelle fair vergleichen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_08/Lecture_A/image_03_03.jpg?v=1784792717" width="250">



>* Gleiche Bedingungen für alle Modelle
>* Trainings- und Testdaten strikt trennen

>* Genauigkeit immer im Kontext bewerten
>* Baselines und Fehlerfolgen mitvergleichen

>* Modellannahmen und Skalierung fair berücksichtigen
>* Mit Validierung stabile, praktische Entscheidungen treffen



In [ ]:
#@title Python-Code - Modelle fair vergleichen

# Wir vergleichen Modelle unter gleichen Bedingungen.
# Ein Testdatensatz macht den Vergleich fair.
# Die Ausgabe zeigt Genauigkeit und Baseline.

import numpy as np
import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Wir nutzen einen kleinen eingebauten Klassifikationsdatensatz.
data = load_breast_cancer()
X = data.data
y = data.target

# Eine einfache Prüfung verhindert unklare Datenfehler.
if X.shape[0] != y.shape[0] or X.shape[0] < 20:
    raise ValueError("Die Daten passen nicht zur Klassifikation.")

# Alle Modelle erhalten dieselbe faire Aufteilung.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

# Skalierung wird nur mit Trainingsdaten gelernt.
models = {
    "Baseline": DummyClassifier(strategy="most_frequent"),
    "Logistische Regression": make_pipeline(
        StandardScaler(), LogisticRegression(max_iter=1000, random_state=42)
    ),
    "k-NN": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
    "Naive Bayes": GaussianNB(),
}

# Jedes Modell wird gleich trainiert und getestet.
results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)
    results.append((name, accuracy))

# Die sortierte Liste macht den Vergleich übersichtlich.
results = sorted(results, key=lambda item: item[1], reverse=True)
print(f"scikit-learn Version: {sklearn.__version__}")
print("Fairer Vergleich: gleiche Trainingsdaten, gleiche Testdaten.")

# Die Baseline zeigt, was ohne echtes Lernen erreichbar ist.
for name, accuracy in results:
    print(f"{name}: Genauigkeit = {accuracy:.3f}")



# <font color="#418FDE" size="6.5" uppercase>**Klassifikation mit NumPy**</font>


In this lecture, you learned to:
- Berechnen Klassifikationsscores, Wahrscheinlichkeiten und binäre Verluste. 
- Trainieren eine kleine logistische Regression mit NumPy. 
- Vergleichen logistische Regression, k-NN, Naive Bayes und Baselines. 

In the next Lecture (Lecture B), we will go over 'Cluster und PCA'